# Cellosaurus — Cell Line Knowledge Resource

**Cellosaurus** is a comprehensive knowledge resource on cell lines maintained by the Swiss Institute of Bioinformatics (SIB). It covers human and animal cell lines, hybridomas, and immortalised lines, providing curated information on species of origin, disease associations, cross-references to other databases, and STR (Short Tandem Repeat) profiles for authentication.

Key data types:
| Data type | Description |
|---|---|
| **Cell line records** | Accession (CVCL_XXXX), name, category, species, and disease context |
| **STR profiles** | Allele calls at standard CODIS loci used for cell line authentication |
| **Cross-references** | Links to ATCC, DSMZ, ECACC, CCLE, DepMap, Cellosaurus, and many others |
| **Synonyms** | Alternative names and previous identifiers |
| **Derived from** | Parent cell line hierarchy (e.g. subclones, transduced variants) |

**Reference:** Bairoch A. (2018), *The Cellosaurus, a Cell-Line Knowledge Resource*, Journal of Biomolecular Techniques

**API base:** `https://api.cellosaurus.org/`

# TODO

* [x] **Ingest data**
    * [x] Connect to Cellosaurus API and confirm access (fetch HeLa CVCL_0030)
    * [x] Search for human cancer cell lines and page through results
    * [x] Parse search results into a Polars DataFrame (accession, name, category, species, disease, sex, age_at_sampling, str_profile_available, cross_reference_count)
    * [x] Download detailed records for a focused set of well-known cell lines (HeLa, HEK293, MCF7, Jurkat, etc.)
    * [x] Parse detailed records into a DataFrame including STR marker alleles
* [ ] **Explore and clean**
    * [ ] Summarise dataset dimensions, missing values, and category/species distributions
    * [ ] Inspect STR profile completeness across cell lines
    * [ ] Examine cross-reference coverage (ATCC, DSMZ, CCLE, DepMap, etc.)
* [ ] **Analysis**
    * [ ] Identify the most common disease contexts and tissue types
    * [ ] Analyse sex and age-at-sampling distributions
    * [ ] Cluster cell lines by STR profile similarity
* [ ] **Visualization**
    * [ ] Bar charts of cell line categories and disease associations
    * [ ] Heatmap of STR allele calls across well-known lines
    * [ ] Cross-reference database coverage plot
* [ ] **Statistical analysis**
    * [ ] Discuss STR profiling mathematics (allele frequency, probability of identity)
    * [ ] Explain multiple hypothesis correction considerations for cell line authentication

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to Cellosaurus API and Confirm Access

In [ ]:
CELLO_BASE = "https://api.cellosaurus.org"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def cello_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the Cellosaurus REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to CELLO_BASE (e.g. "cell-line/CVCL_0030").
    params : dict, optional
        Query parameters appended to the URL.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{CELLO_BASE}/{endpoint}"
    resp = requests.get(url, params={**(params or {}), "format": "json"}, timeout=30)
    resp.raise_for_status()
    time.sleep(0.3)   # polite delay between calls
    return resp.json()


# --- Connectivity check: fetch the canonical HeLa cell line ---
# The API returns {"cell-line": [ {...} ]} even for single-accession lookups
hela_raw = cello_get("cell-line/CVCL_0030")
hela = hela_raw["cell-line"][0]

# API field names (from the Cellosaurus JSON schema):
#   id  = primary name/identifier
#   ca  = category (e.g. "Cancer cell line")
#   sx  = sex (e.g. "Female")
#   ag  = age at sampling (e.g. "30Y")
#   ox  = list of {id, label} for organism (NCBI taxon)
#   di  = list of {id, label} for disease (NCI Thesaurus / ORDO)
name     = hela.get("id", "N/A")
category = hela.get("ca", "N/A")
species  = hela.get("ox", [{}])[0].get("label", "N/A")
disease  = hela.get("di", [{}])[0].get("label", "N/A") if hela.get("di") else "N/A"
sex      = hela.get("sx", "N/A")
age      = hela.get("ag", "N/A")

print(f"Name     : {name}")
print(f"Category : {category}")
print(f"Species  : {species}")
print(f"Disease  : {disease}")
print(f"Sex      : {sex}")
print(f"Age      : {age}")

### 1.2 Search for Human Cancer Cell Lines and Cache Results

In [ ]:
CANCER_CACHE = DATA_DIR / "cellosaurus_cancer_lines.json"
PAGE_SIZE = 100   # maximum rows per request supported by the API


def fetch_cancer_lines(cache_path: Path = CANCER_CACHE) -> list[dict]:
    """
    Search Cellosaurus for human cancer cell lines and page through all results.

    Uses the query ``category:"Cancer cell line" AND species:"Homo sapiens"``
    against the /search/cell-line endpoint.  Results are cached to disk so
    subsequent runs avoid redundant network traffic.

    Parameters
    ----------
    cache_path : Path
        File path for the JSON cache.  Written on first run, read on subsequent runs.

    Returns
    -------
    list[dict]
        Raw cell-line dicts as returned by the API, one entry per cell line.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_lines: list[dict] = []
    start = 0

    # First request to discover total hit count
    resp = cello_get(
        "search/cell-line",
        {"q": 'category:"Cancer cell line" AND species:"Homo sapiens"',
         "start": 0, "rows": 1},
    )
    total = resp.get("total", 0)
    print(f"Total matching cell lines: {total}")

    while start < total:
        resp = cello_get(
            "search/cell-line",
            {"q": 'category:"Cancer cell line" AND species:"Homo sapiens"',
             "start": start, "rows": PAGE_SIZE},
        )
        batch = resp.get("cell-line", [])
        if not batch:
            break
        all_lines.extend(batch)
        start += len(batch)
        print(f"  Fetched {start:>6} / {total}", end="\r")

    print(f"\nDone. Retrieved {len(all_lines)} records.")
    cache_path.write_text(json.dumps(all_lines))
    return all_lines


cancer_lines_raw = fetch_cancer_lines()
print(f"Records in memory: {len(cancer_lines_raw)}")

### 1.3 Flatten Search Results into a Polars DataFrame

In [ ]:
def flatten_cell_line(cl: dict) -> dict:
    """
    Flatten a single raw Cellosaurus cell-line dict into a row-friendly format.

    Parameters
    ----------
    cl : dict
        Raw cell-line record from the Cellosaurus API.  Relevant fields:

        * ``ac``   – accession string, e.g. "CVCL_0030"
        * ``id``   – primary name, e.g. "HeLa"
        * ``ca``   – category string, e.g. "Cancer cell line"
        * ``ox``   – list of organism dicts  [{id, label}]
        * ``di``   – list of disease dicts   [{id, label}]
        * ``sx``   – sex string, e.g. "Female"
        * ``ag``   – age at sampling string, e.g. "30Y"
        * ``str``  – STR profile dict with a "marker" list (may be absent)
        * ``xref`` – list of cross-reference dicts [{db, id}]

    Returns
    -------
    dict
        Flat dict suitable for direct construction of a Polars DataFrame row.
    """
    # STR profile is present when the "str" key contains a non-empty "marker" list
    str_block   = cl.get("str", {}) or {}
    str_markers = str_block.get("marker", []) or []

    # Cross-references: count total links to external databases
    xrefs = cl.get("xref", []) or []

    return {
        "accession":       cl.get("ac"),
        "name":            cl.get("id"),
        "category":        cl.get("ca"),
        "species":         cl.get("ox", [{}])[0].get("label") if cl.get("ox") else None,
        "disease":         cl.get("di", [{}])[0].get("label") if cl.get("di") else None,
        "sex":             cl.get("sx"),
        "age_at_sampling": cl.get("ag"),
        "str_available":   len(str_markers) > 0,   # True if at least one STR locus called
        "xref_count":      len(xrefs),
    }


# Build the DataFrame; Polars infers dtypes from the first batch of rows
rows = [flatten_cell_line(cl) for cl in cancer_lines_raw]
cancer_df = pl.DataFrame(rows).with_columns(
    pl.col("str_available").cast(pl.Boolean),
    pl.col("xref_count").cast(pl.Int32),
)

print(f"Shape  : {cancer_df.shape}")
print(f"Memory : {cancer_df.estimated_size('mb'):.2f} MB")
print(f"\nDtypes :\n{cancer_df.schema}")
cancer_df.head(10)

### 1.4 Fetch Detailed Records for a Panel of Famous Cell Lines

In [ ]:
# Five historically significant human cell lines spanning diverse cancer types
PANEL = {
    "CVCL_0030": "HeLa",    # cervical adenocarcinoma; oldest human cell line (1951)
    "CVCL_0045": "HEK293",  # embryonic kidney; workhorse of molecular biology
    "CVCL_0031": "MCF7",    # breast adenocarcinoma; ER-positive model
    "CVCL_0367": "Jurkat",  # T-cell leukaemia; immune signalling model
    "CVCL_0023": "A549",    # lung carcinoma; respiratory/toxicology model
}


def fetch_panel(panel: dict[str, str]) -> dict[str, dict]:
    """
    Fetch full Cellosaurus records for a set of accessions.

    Parameters
    ----------
    panel : dict[str, str]
        Mapping of accession -> common name (for logging).

    Returns
    -------
    dict[str, dict]
        Mapping of accession -> raw cell-line dict from the API.
    """
    records: dict[str, dict] = {}
    for acc, label in panel.items():
        resp = cello_get(f"cell-line/{acc}")
        cl   = resp["cell-line"][0]
        records[acc] = cl
        print(f"  {acc}  {label:<10}  category={cl.get('ca','?')}")
    return records


panel_raw = fetch_panel(PANEL)
print(f"\nFetched {len(panel_raw)} detailed records.")

### 1.5 Build Detailed DataFrame with STR Markers

In [ ]:
def flatten_panel_record(acc: str, cl: dict) -> dict:
    """
    Flatten a detailed Cellosaurus record into a row with per-locus STR alleles.

    STR (Short Tandem Repeat) profiling is the gold-standard method for cell
    line authentication (ANSI/ATCC ASN-0002).  Each locus is characterised by
    the number of tandem repeats at that genomic position; the result is a
    comma-separated string of allele calls (e.g. "12,14" for two alleles).
    Amelogenin is sex-linked and returns "X" or "X,Y".

    Parameters
    ----------
    acc : str
        Cellosaurus accession (e.g. "CVCL_0030").
    cl : dict
        Raw cell-line dict from the API.

    Returns
    -------
    dict
        Flat dict with metadata fields plus one column per STR locus.
    """
    # --- Metadata ---
    str_block = cl.get("str", {}) or {}
    markers   = str_block.get("marker", []) or []
    xrefs     = cl.get("xref", []) or []

    # Build {locus_name: alleles_string} for every STR marker present
    str_profile: dict[str, str] = {
        m["id"]: m.get("alleles", "")
        for m in markers
        if "id" in m
    }

    # Collect cross-reference databases as a pipe-separated string for inspection
    xref_dbs = " | ".join(sorted({x.get("db", "") for x in xrefs if x.get("db")}))

    row: dict = {
        "accession":        acc,
        "name":             cl.get("id"),
        "category":         cl.get("ca"),
        "species":          cl.get("ox", [{}])[0].get("label") if cl.get("ox") else None,
        "disease":          cl.get("di", [{}])[0].get("label") if cl.get("di") else None,
        "sex":              cl.get("sx"),
        "age_at_sampling":  cl.get("ag"),
        "str_loci_count":   len(markers),
        "xref_count":       len(xrefs),
        "xref_databases":   xref_dbs or None,
        # Include the full comment block as a single string for traceability
        "comments":         cl.get("cc") if isinstance(cl.get("cc"), str) else None,
    }

    # Merge per-locus STR allele calls into the row dict
    row.update({f"str_{locus}": alleles for locus, alleles in str_profile.items()})
    return row


# Collect all rows; each may have different STR loci, so Polars will null-fill missing ones
panel_rows = [flatten_panel_record(acc, cl) for acc, cl in panel_raw.items()]
panel_df   = pl.DataFrame(panel_rows)   # Polars aligns columns; absent loci become null

# Cast numeric-ish metadata columns
panel_df = panel_df.with_columns(
    pl.col("str_loci_count").cast(pl.Int32),
    pl.col("xref_count").cast(pl.Int32),
)

print(f"Shape  : {panel_df.shape}")
print(f"\nDtypes :")
for name, dtype in panel_df.schema.items():
    print(f"  {name:<30} {dtype}")
print()
panel_df